# Prompt 13: fast main run on Kaggle (Version A, T21)

Vaibhav Sharma, MSE2026010. Decision D25: 6 new topics (2 per band), both models in parallel, 1024 tokens.

**Before running (once):**
1. Right panel > Settings: **Accelerator = GPU T4 x2**, **Internet = On** (needs a verified phone number).
2. Add-ons > Secrets: add `OPENALEX_API_KEY` and `CONTACT_EMAIL`, tick both for this notebook.
3. Right panel > Input > Upload: upload `AITR_Lab02_MSE2026010_CiteVerify.zip` as a new dataset (any name).

**Run:** either click *Run All* and keep the tab open (about 25 minutes), or click *Save Version > Save & Run All (Commit)* and close the tab; the result appears in the Output tab of that version.

At the end, download `main_outputs.zip` from the Output panel (`/kaggle/working`).

In [ ]:
# Cell 1: copy the repository from the uploaded dataset into /kaggle/working
import glob, os, shutil, zipfile
os.chdir('/kaggle/working')
src = [p for p in glob.glob('/kaggle/input/**/src', recursive=True) if os.path.isfile(p + '/generate.py')]
if not src:
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall('/kaggle/working/unz')
    src = [p for p in glob.glob('/kaggle/working/unz/**/src', recursive=True) if os.path.isfile(p + '/generate.py')]
root = os.path.dirname(src[0])
if os.path.exists('repo'):
    shutil.rmtree('repo')
shutil.copytree(root, 'repo')
os.chdir('/kaggle/working/repo')
os.makedirs('logs', exist_ok=True)
print(os.getcwd(), sorted(os.listdir('.')))
print('pilot data present:', os.path.exists('data/raw/generations_pilot.jsonl'), os.path.exists('data/topics_frozen.csv'))

In [ ]:
# Cell 2: GPUs and secrets
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
from kaggle_secrets import UserSecretsClient
sc = UserSecretsClient()
for k in ('OPENALEX_API_KEY', 'CONTACT_EMAIL'):
    try:
        os.environ[k] = sc.get_secret(k) or ''
    except Exception:
        os.environ[k] = ''
    print(k, 'set' if os.environ[k] else 'NOT set')

In [ ]:
# Cell 3: install, record the environment, unit tests (expect 36 passed)
!pip install -q -r requirements.in 2>&1 | tail -2
!pip freeze > results/requirements_kaggle.txt
import torch, transformers, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| GPUs', torch.cuda.device_count())
!python -m pytest -q 2>&1 | tee logs/p13_pytest.log | tail -2

In [ ]:
# Cell 4: pin model revisions and freeze the protocol (carried-forward check 2 from Prompt 10)
import re, time
from huggingface_hub import HfApi
ids = ['Qwen/Qwen2.5-7B-Instruct', 'mistralai/Mistral-7B-Instruct-v0.3']
sha = {m: HfApi().model_info(m).sha for m in ids}
now = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
cfgp = 'configs/protocol_v1.yaml'
s = open(cfgp).read()
s = re.sub(r'revisions: \{\}.*', 'revisions: {' + ', '.join(f"'{m}': '{h}'" for m, h in sha.items()) + '}   # pinned ' + now, s)
s = s.replace('frozen: false', 'frozen: true   # frozen ' + now + ' before the main run')
open(cfgp, 'w').write(s)
print(sha)
!grep -n "frozen\|revisions\|threshold\|min_flags\|max_new_tokens\|main_per_band" configs/protocol_v1.yaml | tee logs/p13_frozen_config.log

In [ ]:
# Cell 5: the 6 main topics (fixed by seed 11)
import pandas as pd
from src.generate import subset
t = pd.read_csv('data/topics_frozen.csv')
print(subset(t, 2, 11)[['topic_id', 'topic', 'band', 'works_count']].to_string(index=False))

In [ ]:
# Cell 6: start both models in parallel (one per GPU) and the pilot B2 rerun (fixed code) at the same time
import subprocess
shutil.copy('data/labels/auto_pilot.jsonl', 'data/labels/auto_pilot_before_fix.jsonl')
def start(cmd, log, gpu=None):
    env = dict(os.environ)
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, 'w'), stderr=subprocess.STDOUT)
t0 = time.time()
jobs = {
 'qwen': start('python -m src.generate --models Qwen/Qwen2.5-7B-Instruct --out data/raw/gen_main_qwen.jsonl', 'logs/p13_gen_qwen.log', 0),
 'mistral': start('python -m src.generate --models mistralai/Mistral-7B-Instruct-v0.3 --out data/raw/gen_main_mistral.jsonl', 'logs/p13_gen_mistral.log', 1),
 'pilot_b2': start('python -m src.verify --mode pilot', 'logs/p13_verify_pilot_rerun.log'),
}
print('started', list(jobs))

In [ ]:
# Cell 7: wait for all three (prints progress every minute; about 12 to 18 minutes)
while any(j.poll() is None for j in jobs.values()):
    time.sleep(60)
    done = {k: ('done' if j.poll() is not None else 'running') for k, j in jobs.items()}
    print(f'{(time.time() - t0) / 60:.0f} min', done)
for k, j in jobs.items():
    print(k, 'exit code', j.returncode)
!grep -h " s11:" logs/p13_gen_qwen.log logs/p13_gen_mistral.log
!tail -2 logs/p13_verify_pilot_rerun.log

In [ ]:
# Cell 8: merge the two answer files, parse, check token-limit hits
from src.common import read_jsonl, write_jsonl
g = read_jsonl('data/raw/gen_main_qwen.jsonl') + read_jsonl('data/raw/gen_main_mistral.jsonl')
write_jsonl('data/raw/generations_main.jsonl', g)
print(len(g), 'answers |', sum(r['n_new_tokens'] >= 1024 for r in g), 'hit the 1024-token limit')
!python -m src.parse --mode main 2>&1 | tee logs/p13_parse_main.log
c = pd.DataFrame(read_jsonl('data/parsed/citations_main.jsonl'))
print(pd.crosstab(c['model'], c['parse_status']))

In [ ]:
# Cell 9: B2 on the main citations (t = 0.85 from the config; about 6 minutes)
!python -m src.verify --mode main 2>&1 | tee logs/p13_verify_main.log | tail -3
e = pd.DataFrame(read_jsonl('data/labels/auto_main.jsonl'))
print(e['label'].value_counts().to_dict())
print(e[['seconds', 'api_calls']].mean().round(3).to_dict())

In [ ]:
# Cell 10: blind labelling sheet with every main citation, then package everything
!python scripts/export_labels.py --mode main --n 100000 2>&1 | tee logs/p13_export.log
!cd /kaggle/working/repo && zip -q -r /kaggle/working/main_outputs.zip logs results configs data/raw data/parsed data/labels
!ls -la /kaggle/working/main_outputs.zip
print('Finished in', round((time.time() - t0) / 60, 1), 'minutes after Cell 6. Download main_outputs.zip from the Output panel.')